# Importing Required Libraries

In [1]:
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, TextStreamer
import json
import os

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
os.environ['WANDB_NOTEBOOK_NAME']='llm_finetuning.ipynb'
os.environ["WANDB_PROJECT"] =  "lama3 qlora (local)"

In [3]:
RANDOM_SEED=42

# Loading base model

In [4]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)


==((====))==  Unsloth 2025.4.3: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4060 Ti. Num GPUs = 1. Max memory: 15.602 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"], 
    use_rslora=True,
    use_gradient_checkpointing="unsloth"
)


tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama",
)


Unsloth 2025.4.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


# Preparing data

In [6]:
def format_instruction(example, tokenizer=tokenizer):
    # Formats an instruction tuning example using a tokenizer's chat template.
    
    instruction = example.get('questionText', '').strip()
    structured_plan_dict = example.get('structured_plan')

    if structured_plan_dict and all(key in structured_plan_dict for key in ["Core Issue", "Suggested Coping Strategies", "Immediate Actions", "Long-term Focus Areas"]):
        response_content = ""
        response_content += f"Core Issue: {structured_plan_dict.get('Core Issue', 'Not specified')}\n"
        response_content += f"Suggested Coping Strategies: {structured_plan_dict.get('Suggested Coping Strategies', 'Not specified')}\n"
        response_content += f"Immediate Actions: {structured_plan_dict.get('Immediate Actions', 'Not specified')}\n"
        response_content += f"Long-term Focus Areas: {structured_plan_dict.get('Long-term Focus Areas', 'Not specified')}"
    else:
        response_content = "A detailed plan could not be generated for this request."

    messages = [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": response_content.strip()},
    ]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted_text}


In [7]:
data_file_path = "counselchat_generated_structured_plans_hybrid_gpt35.processed.plans.jsonl"

print(f"Loading data from {data_file_path}...")
training_examples = []
with open(data_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)
        if entry.get('structured_plan') is not None:
            training_examples.append({
                'questionText': entry['questionText'],
                'structured_plan': entry['structured_plan']
            })
print(f"Loaded {len(training_examples)} training examples from the processed file.")


Loading data from counselchat_generated_structured_plans_hybrid_gpt35.processed.plans.jsonl...
Loaded 860 training examples from the processed file.


In [8]:
full_dataset = Dataset.from_list(training_examples)

full_dataset = full_dataset.map(format_instruction, remove_columns=list(full_dataset.features.keys()))

print(f"Formatted dataset created with {len(full_dataset)} examples.")


Map:   0%|          | 0/860 [00:00<?, ? examples/s]

Formatted dataset created with 860 examples.


In [9]:
dataset = full_dataset.train_test_split(test_size=0.1, seed=RANDOM_SEED)

In [10]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 774
    })
    test: Dataset({
        features: ['text'],
        num_rows: 86
    })
})

Let's print some examples.

In [11]:
dataset['train'][0]

{'text': "<|begin_of_text|>[INST] Whenever I leave my girlfriend I get panic attacks. I'm taking medications to control them, but I'm thinking of moving in with her since I get so anxious. [/INST] Core Issue: The user experiences panic attacks whenever they leave their girlfriend and is considering moving in with her as a way to manage the anxiety. The user is currently reliant on medications to control the panic attacks.\nSuggested Coping Strategies: 1. Address the underlying reasons for the panic attacks through talk therapy to understand and work through the root causes.\\n2. Practice coping mechanisms such as breathing exercises and yoga to manage anxiety and stress.\\n3. Explore stress-reducing methods with a trained therapist to develop personalized strategies for anxiety management.\nImmediate Actions: 1. Schedule sessions with a therapist to start exploring the root causes of the panic attacks.\\n2. Begin incorporating breathing exercises and yoga into daily routines to help ma

In [12]:
dataset['test'][5]

{'text': "<|begin_of_text|>[INST] My husband cheated on me and it hurt me very bad. It was a time when my health was poor. I'm have a hard time moving on. [/INST] Core Issue: The user is struggling to move on after experiencing betrayal from her husband who cheated on her during a time of poor health.\nSuggested Coping Strategies: 1. Communicate openly with the husband to understand his feelings, regret, and reasons behind the betrayal.\\n2. Consider seeking marriage counseling to navigate the trust rebuilding process and address underlying issues.\\n3. Allow yourself time to process your emotions and acknowledge that healing from betrayal takes time.\\n4. Assess the relationship dynamics and evaluate the husband's actions post-betrayal to make informed decisions.\nImmediate Actions: 1. Initiate a candid conversation with your husband to discuss his feelings, reasons for cheating, and steps towards rebuilding trust.\\n2. Consider seeking professional help from a marriage counselor to f

# Training Config

<!-- # training_args = SFTConfig(
#     output_dir="./results",
#     num_train_epochs=1,
#     run_name="llama3-8b-mental-health-qlora",
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=16,
#     optim="paged_adamw_32bit",
#     save_steps=50, 
#     logging_steps=10,
#     learning_rate=1e-4, 
#     weight_decay=0.01, 
#     fp16=False,
#     bf16=torch.cuda.is_bf16_supported(), 
#     max_grad_norm=0.3,
#     max_steps=-1,
#     warmup_ratio=0.03,
#     group_by_length=True,
#     lr_scheduler_type="cosine",
#     report_to="wandb",
#     eval_strategy="steps",
#     save_strategy="steps",
#     eval_steps=50, 
#     load_best_model_at_end=True, 
#     metric_for_best_model="eval_loss", 
#     greater_is_better=False, 
#     dataset_text_field="text",
#     max_seq_length=512,
#     label_names=["labels"] 
# ) -->

In [13]:
training_args = SFTConfig(
        run_name='finetuning_lora',
        learning_rate=1e-4,
        lr_scheduler_type="linear",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=2,
        eval_strategy="steps",
        do_eval=True,
        eval_steps=0.1,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        output_dir="output",
        seed=0,

)

# Start Training

In [14]:
trainer=SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=training_args
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/774 [00:00<?, ? examples/s]

Unsloth: Hugging Face's packing is currently buggy - we're disabling it for now!


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/86 [00:00<?, ? examples/s]

Unsloth: Hugging Face's packing is currently buggy - we're disabling it for now!


In [15]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 774 | Num Epochs = 2 | Total steps = 96
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)
wandb: Currently logged in as: nikhilcramakrishnan to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,1.387100,1.466457
20,1.254700,1.328930
30,1.272200,1.285607
40,1.253000,1.258705
50,1.204700,1.237900
60,1.081300,1.229488
70,1.112300,1.226162
80,1.181300,1.221014
90,1.194800,1.218500


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=96, training_loss=1.2566126088301341, metrics={'train_runtime': 1303.1387, 'train_samples_per_second': 1.188, 'train_steps_per_second': 0.074, 'total_flos': 3.897625111068672e+16, 'train_loss': 1.2566126088301341})

# Evaluating

In [16]:
test_prompt = """
Hey everyone, I'm really struggling right now and hoping maybe someone here can offer some advice. 
For weeks now, I've been battling insomnia – some nights I can barely fall asleep,
others I wake up in the early hours and can't drift back off, and I'm constantly exhausted and foggy-brained.
It's starting to affect everything: my work, my mood, my relationships, and I'm feeling pretty desperate and isolated about it.
"""

In [17]:
messages = [{"role": "user", "content": test_prompt}]
model_input = tokenizer.apply_chat_template(messages, tokenize=False, add_special_tokens=True)

inputs = tokenizer(model_input, return_tensors="pt").to("cuda")

try:
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\nPrompt: {test_prompt}")
    print(f"\nModel response:\n{response}")
except Exception as e:
    print(f"Error during inference: {e}")


Prompt: 
Hey everyone, I'm really struggling right now and hoping maybe someone here can offer some advice. 
For weeks now, I've been battling insomnia – some nights I can barely fall asleep,
others I wake up in the early hours and can't drift back off, and I'm constantly exhausted and foggy-brained.
It's starting to affect everything: my work, my mood, my relationships, and I'm feeling pretty desperate and isolated about it.


Model response:
 Core Issue: The user is struggling with insomnia, leading to exhaustion, foggy brain, and affecting various aspects of their life, including work, mood, and relationships. They are feeling desperate and isolated due to the impact of sleeplessness on their well-being.
Suggested Coping Strategies: 1. Seek a sleep study to rule out any underlying medical issues contributing to the insomnia.\n2. Consider taking melatonin, a natural sleep hormone, to help regulate the sleep-wake cycle.\n3. Practice relaxation techniques such as deep breathing, progr

In [18]:
model.save_pretrained_merged("model", tokenizer, save_method="merged_16bit")
model.push_to_hub_merged("counsel_assist", tokenizer, save_method="merged_16bit")

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 8.88 out of 30.27 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 44%|████▍     | 14/32 [00:00<00:00, 18.24it/s]
We will save to Disk and not RAM now.
100%|██████████| 32/32 [00:19<00:00,  1.62it/s]


Unsloth: Saving tokenizer... Done.
Done.
Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 9.42 out of 30.27 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 32/32 [00:11<00:00,  2.86it/s]


Unsloth: Saving to organization with address nikhil-c-r/counsel_assist
Unsloth: Saving tokenizer... Done.
Unsloth: Saving to organization with address nikhil-c-r/counsel_assist
Unsloth: Uploading all files... Please wait...


...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Done.
Saved merged model to https://huggingface.co/None/counsel_assist
